## 문서 파싱 

In [2]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [3]:
SAMPLES = SANDBOX / "samples"  # 실습 재료 배치할 폴더 

In [4]:
# 파일,폴더를 통째로 복사/이동하는 표준 라이브러리 
import shutil 

# 다운로드 받은 경로, 프로젝트 폴더 옆에 seed_docs 압축해제한 경우 
CANDIDATES = [pathlib.Path.home() / 'Downloads' / 'seed_docs', ROOT.parent / 'seed_docs']

SAMPLES.mkdir(parents=True, exist_ok=True) # 샘플 폴더 없으면 생성 
found = [p for p in SAMPLES.rglob('*') if p.is_file()] # 파일들 조회 

# 이미 파일들이 있으면, 몇개인지 출력 
if found:
    print(f'이미 있습니다 : {SAMPLES.relative_to(ROOT)}')
    print(f'파일 {len(found)}개')
else: # 파일들이 없으면 복사 
    SRC = next((c for c in CANDIDATES if c.is_dir()), None)
    if SRC is None:
        print('재료 폴더를 찾지 못했습니다. 아래 두 자리 중 한 곳에 풀어 주세요 —')
        for c in CANDIDATES:
            print('   ', c)
    else:
        shutil.copytree(SRC, SAMPLES, dirs_exist_ok=True)
        print(f'복사 완료 : {SAMPLES.relative_to(ROOT)}')
        print(f"파일 {sum((1 for p in SAMPLES.rglob('*') if p.is_file()))}개")


이미 있습니다 : sandbox/w5/day01/samples
파일 30개


In [5]:
from collections import Counter  # 항목별 개수를 세는 표준 라이브러리 

ORDER = [".docx", ".pdf", ".hwpx"]

# 파일들만 모아서 확장자(소문자변환) 기준으로 카운트 
counts = Counter(p.suffix.lower() for p in SAMPLES.rglob("*") if p.is_file())
others = sum(n for ext, n in counts.items() if ext not in ORDER)

print(f"파일 {sum(counts.values())}개")
print(" " + " , ".join(f"{ext} {counts[ext]}" for ext in ORDER) + f" , 그 밖 {others}")


파일 30개
 .docx 10 , .pdf 11 , .hwpx 3 , 그 밖 6


* 참고 : 맥에서 파일명이 NFC 인지 체크하는 코드

In [6]:
import unicodedata

fs = [p.name for p in SAMPLES.glob("*.pdf")]
print(len(fs), "개")

for n in fs[:3]:
    print(n, unicodedata.is_normalized("NFC", n))


10 개
DOC-SE-003_정보보안_지침_v2.2.pdf True
DOC-SE-011_장비_반출입_절차_v1.0.pdf True
DOC-PM-005_프로젝트_관리_지침_v2.0.pdf True


* PDF 

In [7]:
from pypdf import PdfReader

# Path.glob("패턴") : 폴더 안에서 이름이 패턴에 맞는 파일을 찾아 하나씩 내놓는 객체 
# next(...) : 조회된 파일들 중 첫번째 하나만 꺼낸다. 
PDF = next(SAMPLES.glob("DOC-HR-014_*_v2.0.pdf"))

reader = PdfReader(str(PDF)) # 파일 열기, 경로 문자열로 넘기기 
print("쪽수 : ", len(reader.pages))


쪽수 :  16


In [8]:
first = reader.pages[0].extract_text()

print(repr(first[:60])) # repr() : 화면에 안보이는 것까지 그대로 보여준다. \n, 앞뒤 공백 등 
print(first[:60])

'일반\n 국내출장 여비 규정\n v2.0\n 문서번호 DOC-HR-014\n 시행일 2025-07-01\n 소관부서 '
일반
 국내출장 여비 규정
 v2.0
 문서번호 DOC-HR-014
 시행일 2025-07-01
 소관부서 


* 페이지 단위로 읽기

In [15]:
total = 0 

for page_no, page in enumerate(reader.pages, 1):
    text = (page.extract_text() or "").strip()
    total += len(text)
    print(f"p.{page_no}  {len(text)}자")

print("-"*15)
print(f"{len(reader.pages)} 페이지, 합계 약 {round(total, -2):,} 자")

p.1  108자
p.2  231자
p.3  443자
p.4  272자
p.5  1070자
p.6  903자
p.7  1005자
p.8  1042자
p.9  961자
p.10  222자
p.11  281자
p.12  282자
p.13  310자
p.14  244자
p.15  211자
p.16  216자
---------------
16 페이지, 합계 약 7,800 자


In [16]:
SCAN = SAMPLES / "_formats" / "sample_scanned.pdf"

scan_reader = PdfReader(str(SCAN))
print("쪽수 : ", len(scan_reader.pages))

total = 0

for page_no, page in enumerate(scan_reader.pages, 1):
    text = (page.extract_text() or "").strip()
    total += len(text)
    print(f"p.{page_no}  {len(text)}자")

print("-"*15)
print(f"{len(scan_reader.pages)} 페이지, 합계 약 {round(total, -2):,} 자")

쪽수 :  1
p.1  0자
---------------
1 페이지, 합계 약 0 자


In [17]:
from app.core.logging import get_logger

log = get_logger('app.rag.local_parsers')

blocks = []
for page_no, page in enumerate(scan_reader.pages, 1):
    text = (page.extract_text() or '').strip()
    if not text:
        log.info('텍스트 없음 (스캔본으로 보임): %s p.%d', SCAN.name, page_no)
        continue
    blocks.append(text)
print(f'블록 {len(blocks)}개')

09:25:11 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
블록 0개


In [18]:
from app.rag.local_parsers import parse_pdf

doc = parse_pdf(PDF)

print(f'블록 {len(doc.blocks)} · page_count {doc.page_count} · table_count {doc.table_count}')
sixth = doc.blocks[5] # 6페이지 확인 
print(f'여섯 번째 블록 : kind={sixth.kind}  locator={sixth.locator}')
print()
print('※ ImportError 가 나면 Kernel ▸ Restart Kernel 뒤 맨 위부터 다시 실행하세요.')

블록 16 · page_count 16 · table_count 0
여섯 번째 블록 : kind=조항  locator=p.6

※ ImportError 가 나면 Kernel ▸ Restart Kernel 뒤 맨 위부터 다시 실행하세요.


In [19]:
# 스캔 PDF도 테스트 -> 로그 나오는지 확인하기 
doc = parse_pdf(SCAN)
print(f"블럭 : {len(doc.blocks)} / page_count : {doc.page_count} / table_count : {doc.table_count}")

09:25:11 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
블럭 : 0 / page_count : 1 / table_count : 0


* DOCX

In [20]:
import zipfile  # 압축 파일을 여는 표준 라이브러리 

# 워드 문서 조회 
DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
print('첫 4바이트 :', DOCX.read_bytes()[:4], '  ← ZIP 표식') # b'PK' = zip 파일 

# 압축 해제 
with zipfile.ZipFile(DOCX) as zf:
    names = zf.namelist()    # 압축 안에 든 파일 이름 목록 추출 
print(f'엔트리 {len(names)}개')

for name in names[:6]:
    mark = '  ← 본문이 여기 있습니다' if name == 'word/document.xml' else ''
    print(f'  {name}{mark}')


첫 4바이트 : b'PK\x03\x04'   ← ZIP 표식
엔트리 17개
  [Content_Types].xml
  _rels/.rels
  docProps/core.xml
  docProps/app.xml
  word/document.xml  ← 본문이 여기 있습니다
  word/_rels/document.xml.rels


In [21]:
from docx import Document

doc = Document(str(DOCX))

print(f"문단 : {len(doc.paragraphs)}개")
print(f"표 : {len(doc.tables)}개")
print("첫 문단 : ", doc.paragraphs[0].text)

문단 : 246개
표 : 7개
첫 문단 :  일반


*빈 문단을 빼고, 진짜 글자가 있는 문단만 남기기*

In [22]:
texts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

print(f"글자가 있는 문단 개수 : {len(texts)}개")
print(f"빈 문단 : {len(doc.paragraphs) - len(texts)}개")  # 문서에서 엔터만 친 공백들 
print(texts[:3])

글자가 있는 문단 개수 : 226개
빈 문단 : 20개
['일반', '국내출장 여비 규정', 'v2.0']


In [23]:
# 문단 먼저 읽고, 표는 나중에 뒤에 이어 붙이는 방식 

# 문단 읽기 
bad = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

table_pos = []

# 테이블 내부의 글자 읽어서 이어 붙이기 
for table in doc.tables:
    cells = [cell.text.strip() for row in table.rows for cell in row.cells]
    bad.append(' '.join((c for c in cells if c)))
    table_pos.append(len(bad))
    
print(f'총 블록 {len(bad)}개')
print(f'표가 들어간 자리 : {table_pos[0]} ~ {table_pos[-1]} 번째  (전부 맨 끝)')


총 블록 233개
표가 들어간 자리 : 227 ~ 233 번째  (전부 맨 끝)


In [24]:
only_paragraphs = sum((len(t) for t in texts))

table_chars = sum((len(cell.text.strip()) for table in doc.tables for row in table.rows for cell in row.cells))

with_tables = only_paragraphs + table_chars

print(f'문단만 읽었을 때 : {only_paragraphs:,}자')
print(f'표까지 읽었을 때 : {with_tables:,}자')
print('─' * 29)
print(f'표 안 글자 {table_chars:,}자  =  전체의 {table_chars / with_tables:.1%}')
print()
print(f'우리 코드에서 첫 표의 자리 : {table_pos[0]} 번째')


문단만 읽었을 때 : 6,230자
표까지 읽었을 때 : 7,358자
─────────────────────────────
표 안 글자 1,128자  =  전체의 15.3%

우리 코드에서 첫 표의 자리 : 227 번째


* 본문 xml 을 원문 순서대로 훑기 

In [25]:
from collections import Counter

body = doc.element.body # 본문 xml 요소 
tags = []

for child in body.iterchildren(): # 바로 아래 자식들을 원문에 놓인 순서 그대로 
    tag = child.tag.split('}')[-1] # child.tag "{긴 주소}p"
    # print("tag :", child.tag)
    if tag not in ('p', 'tbl'):  # 문단과 표가 아니면 넘어가기 
        continue
    if tag == 'p' and (not ''.join(child.itertext()).strip()): # 빈 문단 제외 
        continue
    tags.append(tag)
kind = Counter(tags)
print(f'본문 자식 {len(tags)}개')
print(f"종류 : p {kind['p']} · tbl {kind['tbl']}")
print('tbl 위치 :', [i for i, t in enumerate(tags, 1) if t == 'tbl'])


본문 자식 233개
종류 : p 226 · tbl 7
tbl 위치 : [10, 218, 221, 224, 227, 229, 232]


In [26]:
from docx.table import Table
from docx.text.paragraph import Paragraph

blocks = []
tables = 0

for child in body.iterchildren():
    tag = child.tag.split("}")[-1]
    if tag == "p":
        text = Paragraph(child, doc).text.strip()  
        if text:
            blocks.append(text)
    elif tag == "tbl":
        table = Table(child, doc) 
        tables += 1 
        blocks.append(f"[표{tables}]") # 일단 표 번호만 추가 (자리만 잡아두기)

print(f"블록 {len(blocks)}개")
print("10번째 항목 : ", blocks[9])
print("9번째 항목 : ", blocks[8])

블록 233개
10번째 항목 :  [표1]
9번째 항목 :  제·개정 이력


In [27]:
path = ROOT / 'backend' / 'app' / 'rag' / 'local_parsers.py'
text = path.read_text(encoding='utf-8')
print(path.name, '·', len(text.splitlines()), '줄')
for name in ['parse_pdf', 'parse_docx']:
    print(f'  def {name:<11}:', text.count(f'def {name}('), '개')

local_parsers.py · 64 줄
  def parse_pdf  : 1 개
  def parse_docx : 1 개


In [28]:
try:
    from app.rag.local_parsers import parse_docx
except ImportError as err:
    print('import 실패 :', err)
    print('→ Kernel ▸ Restart 로 커널을 재시작한 뒤 맨 위 두 셀부터 다시 실행하세요.')
    raise
    
doc = parse_docx(DOCX)
print(f'블록 {len(doc.blocks)} · page_count {doc.page_count} · table_count {doc.table_count}')
print('표 위치(0부터) :', [i for i, b in enumerate(doc.blocks) if b.kind == '표'])
print("(이전의 '10번째' 와 1 차이가 나는 것은 0 부터 세기 때문입니다)")

블록 233 · page_count 1 · table_count 7
표 위치(0부터) : [9, 217, 220, 223, 226, 228, 231]
(이전의 '10번째' 와 1 차이가 나는 것은 0 부터 세기 때문입니다)
